In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pipeline import PipelineConfig, run_pipeline, plot_pipeline_results

In [ ]:
"""Data Loading"""

# selecting variables and dates
columns_wanted = ["lnipsa", "lncpi", "r-mkt", "lndem"]
start_date = "1987-01"
end_date = "1998-12"
# setting lags, number of variables, etc
L = 6
C = 5
T = len(pd.date_range(start=start_date, end=end_date, freq="MS"))
N = 4
N_w = 3
N_z = 3
L_w = [0,1]
L_z1 = [1,2]
L_z2 = [0,1]
K = N*L + N_w*len(L_w)
n_lag_z = len(L_z1)  # == len(L_z2)
Z_width = N_z * n_lag_z

# loading data
def load_and_filter(path, columns_wanted):
    df = pd.read_csv(path, index_col=0)
    df.index = pd.to_datetime(df.index, format="%Y-%m")
    df = df[columns_wanted]
    load_start = pd.Timestamp(start_date) - pd.DateOffset(months=L)
    df = df.loc[load_start:end_date]
    return df

fin = load_and_filter("Jarocinski Data/mj-data/2008-euro/finland.csv", columns_wanted)
fra = load_and_filter("Jarocinski Data/mj-data/2008-euro/france.csv", columns_wanted)
ita = load_and_filter("Jarocinski Data/mj-data/2008-euro/italy.csv", columns_wanted)
por = load_and_filter("Jarocinski Data/mj-data/2008-euro/portugal.csv", columns_wanted)
spa = load_and_filter("Jarocinski Data/mj-data/2008-euro/spain.csv", columns_wanted)

wor = load_and_filter("Jarocinski Data/mj-data/2008-world/world1.csv", ["ffr", "lnoil-eur", "lnnfd-eur"])
ger1 = load_and_filter("Jarocinski Data/mj-data/2008-world/world1.csv", ["lnipsa-ger"])
ger2 = load_and_filter("Jarocinski Data/mj-data/2008-world/world1.csv", ["r-mkt-ger", "lneur-usd"])

Y = np.stack([fin.values, fra.values, ita.values, por.values, spa.values], axis=0)
W = wor.values
Z1 = ger1.values
Z2 = ger2.values

In [ ]:
config_real = PipelineConfig(
    name="real_data",
    country_names=["Finland", "France", "Italy", "Portugal", "Spain"],
    variable_names=["Output", "Prices", "Interest rate", "Exchange rate"],
)

results_real = run_pipeline(
    Y, W, Z1, Z2, C, N, N_w, T, K, Z_width, L, L_w, L_z1, L_z2,
    config=config,
)

plot_pipeline_results(results_real)

In [ ]:
results_gibbs = results_real["gibbs"]["results"]

In [ ]:
N = 4
N_w = 3
N_z = 3
L_w = [0,1]
L_z1 = [1,2]
L_z2 = [0,1]
K = N*L + N_w*len(L_w)
n_lag_z = len(L_z1)
Z_width = N_z * n_lag_z

T_held = 144
C_held = 5
Ts = [30, 300]
Cs = [3, 10]

In [ ]:
def sample_var_y(var_y_real, C, rng):
    """Draw (C, N) variances for simulated countries, matched in scale/spread
    to the real data's per-country variable variances (var_y_real: (5, N))."""
    log_var = np.log(var_y_real)
    mean_log, std_log = log_var.mean(axis=0), log_var.std(axis=0)
    return np.exp(rng.normal(mean_log, std_log, size=(C, len(mean_log))))


def build_lambda_c(var_y, target_var_w, N, N_w, L, L_w, K):
    """var_y: (C, N) -- variances per simulated country.
    target_var_w: (N_w,) -- variance of the (simulated) W series."""
    C = var_y.shape[0]
    Lambda = np.zeros((C, N*K, N*K))
    var_index = ([n for l in range(L) for n in range(N)] +
                 [N + j for l in range(len(L_w)) for j in range(N_w)])
    for c in range(C):
        var_all = np.append(var_y[c], target_var_w)
        diag = np.array([var_y[c][n] / var_all[var_index[k]]
                          for n in range(N) for k in range(K)])
        Lambda[c] = np.diag(diag)
    return Lambda

In [ ]:
rng = np.random.default_rng(seed)

# from real data, computed once
var_y_real = np.array([np.var(Y[c], axis=0) for c in range(5)])   # (5, N)

# for the current simulation setting (any C)
var_y = sample_var_y(var_y_real, C, rng)                          # (C, N)
target_var_w = np.var(W_sim, axis=0)                              # (N_w,), from simulated W

Lambda_sim = build_lambda_c(var_y, target_var_w, N, N_w, L, L_w, K)

beta0_sim = np.mean(results_gibbs['beta0'], axis=0)

lambda_sim = np.mean(results_gibbs['lam'])

In [ ]:
# once, from real data
Sigma_c_real = np.array([results_gibbs["Sigma_c"][c].mean(axis=0) for c in range(5)])  # (5, N, N)

# average correlation structure across the 5 real countries
corr_real = np.array([
    Sigma_c_real[c] / np.outer(np.sqrt(np.diag(Sigma_c_real[c])), np.sqrt(np.diag(Sigma_c_real[c])))
    for c in range(5)
])
target_corr = corr_real.mean(axis=0)  # (N, N), shared correlation pattern

# per-country variances (any C), same machinery as var_y
sigma_diag_real = np.array([np.diag(Sigma_c_real[c]) for c in range(5)])  # (5, N)
sigma_diag = sample_var_y(sigma_diag_real, C, rng)                        # (C, N)

Sigma = np.zeros((C, N, N))
for c in range(C):
    D = np.diag(np.sqrt(sigma_diag[c]))
    Sigma[c] = D @ target_corr @ D

In [ ]:
def fit_ar1(x):
    x = np.asarray(x)
    x_t, x_lag = x[1:], x[:-1]
    phi = np.sum(x_lag * x_t) / np.sum(x_lag**2)
    resid = x_t - phi * x_lag
    sigma = np.std(resid)
    return phi, sigma

def simulate_ar1(phi, sigma, T_total, rng, x0=0.0):
    x = np.zeros(T_total)
    x[0] = x0
    eps = rng.normal(0, sigma, size=T_total)
    for t in range(1, T_total):
        x[t] = phi * x[t-1] + eps[t]
    return x

def simulate_exog(real_series, T_total, rng):
    # real_series: (T_real, n_cols)
    n_cols = real_series.shape[1]
    sim = np.zeros((T_total, n_cols))
    for j in range(n_cols):
        phi, sigma = fit_ar1(real_series[:, j])
        sim[:, j] = simulate_ar1(phi, sigma, T_total, rng)
    return sim

brun = 50
T_total = T + L + burn
W_sim  = simulate_exog(W,  T_total, rng)
Z1_sim = simulate_exog(Z1, T_total, rng)
Z2_sim = simulate_exog(Z2, T_total, rng)

In [ ]:
def simulate_panel(W, Z1, Z2, beta0, Lambda, lam,
                    C, N, N_w, T, L, L_w, L_z1, L_z2, K, Z_width,
                    sigma_scale=0.1, burn=50, seed=None):
    rng = np.random.default_rng(seed)
    T_keep = T + L
    T_total = T_keep + burn

    betas = np.zeros((C, N, K))
    for c in range(C):
        beta_vec = beta0 + np.sqrt(lam * np.diag(Lambda[c])) * rng.standard_normal(N*K)
        betas[c] = beta_vec.reshape(N, K)

        Comp = np.zeros((N*L, N*L))
        Comp[:N, :] = betas[c, :, :N*L]
        Comp[N:, :-N] = np.eye(N*(L-1))
        radius = np.max(np.abs(np.linalg.eigvals(Comp)))
        if radius >= 1:
            betas[c, :, :N*L] *= 0.95 / radius

    gamma_z1 = rng.normal(0, 0.2, size=(C, N, Z_width))
    gamma_z2 = rng.normal(0, 0.2, size=(C, N, Z_width))

    Sigma = np.zeros((C, N, N))
    innovations = np.zeros((T_total, C, N))
    for c in range(C):
        A = rng.standard_normal((N, N)) * sigma_scale
        Sigma[c] = A @ A.T + np.eye(N)*0.01
        innovations[:, c, :] = rng.multivariate_normal(np.zeros(N), Sigma[c], size=T_total)

    assert len(W) == T_total and len(Z1) == T_total and len(Z2) == T_total

    Y = np.zeros((C, T_total, N))
    for t in range(L, T_total):
        exog_w = np.concatenate([W[t-l] for l in L_w])
        exog_z1 = np.concatenate([Z1[t-l] for l in L_z1])
        exog_z2 = np.concatenate([Z2[t-l] for l in L_z2])
        for c in range(C):
            lags_y = np.concatenate([Y[c, t-l, :] for l in range(1, L+1)])
            regressors = np.concatenate([lags_y, exog_w])
            Y[c, t] = (betas[c] @ regressors
                       + gamma_z1[c] @ exog_z1
                       + gamma_z2[c] @ exog_z2
                       + innovations[t, c])

    Y = Y[:, burn:, :]

    ground_truth = {"beta_c": betas, "gamma_z1": gamma_z1, "gamma_z2": gamma_z2, "Sigma_c": Sigma}
    return Y, ground_truth